In [1]:
# =============================================================================
# CELL F1 — FLAML pilot: environment, same data and splits, API check
# Kernel: .venv-flaml (Python 3.10). Restart the kernel before running.
# =============================================================================
import os
for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "VECLIB_MAXIMUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[_v] = "1"

import sys, json, time, hashlib, platform, warnings
from dataclasses import dataclass
from pathlib import Path
import numpy as np, pandas as pd, sklearn, shap, flaml
from flaml import AutoML
warnings.filterwarnings("ignore")

PROJECT = Path("/home/dbanerjee/example_for_debjeet")
FL_ROOT = Path("/ds-dsa/dbanerjee/automl_flaml")
OUT = FL_ROOT / "results"
DIRS = {k: OUT / k for k in ("data", "xai", "metrics", "figures", "tmp")}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)
DATA_DIR, P2_DATA = PROJECT / "datasets", PROJECT / "results" / "phase1" / "data"

print({"python": platform.python_version(), "flaml": flaml.__version__,
       "scikit-learn": sklearn.__version__, "shap": shap.__version__}, "\nkernel:", sys.executable)
assert ".venv-flaml" in sys.executable or "automl_flaml" in sys.executable, "wrong kernel selected"

# ---- configuration and seeds: identical to the auto-sklearn study -------------------------
@dataclass(frozen=True)
class Config:
    dataset_ids: tuple = (37, 31)            # pilot: diabetes, credit-g
    master_seed: int = 20260923
    n_background: int = 100
    n_explain: int = 200
    n_permutations: int = 10
    n_train_seeds: int = 3
    k_configs: int = 5
    def max_evals(self, d): return self.n_permutations * (2 * d + 1)
CFG = Config()

def derive_seed(*keys):
    payload = json.dumps([CFG.master_seed, *map(str, keys)]).encode()
    return int.from_bytes(hashlib.sha256(payload).digest()[:4], "little") % (2**31 - 1)

# ---- same raw data and Phase-2 splits ---------------------------------------------------------
EVENT_LABEL = {37: "tested_positive", 31: "bad"}
@dataclass
class Data:
    did: int; X: np.ndarray; feature_names: list; cat_mask: np.ndarray; y: np.ndarray
    @property
    def d(self): return self.X.shape[1]

def load_raw(did):
    folder = DATA_DIR / str(did)
    Xdf = pd.read_csv(folder / "features.csv")
    t = pd.read_csv(folder / "targets.csv").iloc[:, 0]
    cat_ind = json.load(open(folder / "categorical_indicator.json"))
    cols = list(Xdf.columns)
    cat = np.array([cat_ind[c] is True for c in cols])
    X = np.column_stack([Xdf[c].astype("category").cat.codes.to_numpy() if cat[j]
                         else pd.to_numeric(Xdf[c]).to_numpy() for j, c in enumerate(cols)]).astype(float)
    return Data(did, X, cols, cat, (t == EVENT_LABEL[did]).to_numpy().astype(int))

DATA = {did: load_raw(did) for did in CFG.dataset_ids}
ROWS = {did: np.load(P2_DATA / f"rowsets_{did}.npz", allow_pickle=True) for did in CFG.dataset_ids}

def to_frame(did, X):
    """Raw-feature matrix -> DataFrame with categoricals as category dtype (FLAML prepares it)."""
    D = DATA[did]
    df = pd.DataFrame(X, columns=D.feature_names)
    for j, c in enumerate(D.feature_names):
        if D.cat_mask[j]:
            df[c] = pd.Categorical(df[c].astype(int), categories=list(range(int(D.X[:, j].max()) + 1)))
    return df

for did in CFG.dataset_ids:
    r = ROWS[did]
    print(f"  {did}: train {len(r['train_idx'])}, test {len(r['test_idx'])}, E {len(r['E_idx'])}, "
          f"B {r['bg_idx'].shape}, categorical {int(DATA[did].cat_mask.sum())} of {DATA[did].d}")

# ---- API check 1: learners and their search spaces ----------------------------------------------
automl = AutoML()
learners = ["lgbm", "xgboost", "xgb_limitdepth", "rf", "extra_tree", "catboost", "lrl1", "lrl2", "kneighbor"]
did = 37
n, d = len(ROWS[did]["train_idx"]), DATA[did].d
spaces = {}
for name in learners:
    try:
        cls = automl._state.learner_classes.get(name) if hasattr(automl, "_state") else None
        if cls is None:
            from flaml.automl import model as M
            cls = {"lgbm": M.LGBMEstimator, "xgboost": M.XGBoostSklearnEstimator,
                   "xgb_limitdepth": M.XGBoostLimitDepthEstimator, "rf": M.RandomForestEstimator,
                   "extra_tree": M.ExtraTreesEstimator, "catboost": M.CatBoostEstimator,
                   "lrl1": M.LRL1Classifier, "lrl2": M.LRL2Classifier, "kneighbor": M.KNeighborsEstimator}[name]
        sp = cls.search_space(data_size=(n, d), task="binary")
        spaces[name] = sp
        print(f"  {name:<15} hyperparameters: {list(sp)}")
    except Exception as e:
        print(f"  {name:<15} NOT AVAILABLE: {type(e).__name__}: {str(e)[:100]}")

# ---- API check 2: sample one random configuration from a search space ------------------------------
rng = np.random.RandomState(derive_seed("flaml-check"))
def sample_config(space, rng):
    cfg = {}
    for hp, spec in space.items():
        dom = spec.get("domain")
        cfg[hp] = dom.sample(random_state=rng) if hasattr(dom, "sample") else dom
    return cfg
cfg = sample_config(spaces["lgbm"], rng)
print("\nrandom lgbm configuration:", cfg)

# ---- API check 3: train one FLAML pipeline with that FIXED configuration ----------------------------
r = ROWS[did]
Xtr, ytr = to_frame(did, DATA[did].X[r["train_idx"]]), DATA[did].y[r["train_idx"]]
t0 = time.time()
aml = AutoML()
aml.fit(X_train=Xtr, y_train=ytr, task="classification", metric="log_loss",
        estimator_list=["lgbm"], starting_points={"lgbm": cfg}, max_iter=1,
        retrain_full=True, seed=derive_seed("flaml-check", 0), n_jobs=1, verbose=0)
p = aml.predict_proba(to_frame(did, DATA[did].X[r["E_idx"]]))[:, 1]
print(f"\ntrained in {time.time() - t0:.1f}s | learner {aml.best_estimator} | config used {aml.best_config}")
print(f"P(y=1) on E in [{p.min():.3f}, {p.max():.3f}]")
print("config used == config requested:", all(aml.best_config.get(k) == v for k, v in cfg.items()))

/home/dbanerjee/example_for_debjeet/.venv-flaml/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'python': '3.10.12', 'flaml': '2.7.0', 'scikit-learn': '1.7.2', 'shap': '0.49.1'} 
kernel: /home/dbanerjee/example_for_debjeet/.venv-flaml/bin/python
  37: train 537, test 231, E 200, B (5, 100), categorical 0 of 8
  31: train 700, test 300, E 200, B (5, 100), categorical 13 of 20
  lgbm            hyperparameters: ['n_estimators', 'num_leaves', 'min_child_samples', 'learning_rate', 'log_max_bin', 'colsample_bytree', 'reg_alpha', 'reg_lambda']
  xgboost         hyperparameters: ['n_estimators', 'max_leaves', 'min_child_weight', 'learning_rate', 'subsample', 'colsample_bylevel', 'colsample_bytree', 'reg_alpha', 'reg_lambda']
  xgb_limitdepth  hyperparameters: ['n_estimators', 'max_depth', 'min_child_weight', 'learning_rate', 'subsample', 'colsample_bylevel', 'colsample_bytree', 'reg_alpha', 'reg_lambda']
  rf              NOT AVAILABLE: AttributeError: 'str' object has no attribute 'is_classification'
  extra_tree      NOT AVAILABLE: AttributeError: 'str' object has no attribute 'is_cl

In [3]:
# =============================================================================
# CELL F2 — FLAML pilot: design (learner x balancing x 5 configs from FLAML's own
#           search spaces), train with 3 seeds, explain exactly like Phase 2,
#           5-fold CV (seed 0) for Rashomon sets. Isolated, resumable jobs.
#   ALL outputs go to /ds-dsa/dbanerjee/automl_flaml/results/
#     data/ xai/ metrics/ figures/ tmp/ work/
#   Updates: 64 GB memory cap (CatBoost threads); interrupted jobs are never saved;
#            crashed / timed-out / memout results from earlier runs are removed and redone.
# =============================================================================
import pickle, importlib
from joblib import Parallel, delayed
from flaml.automl import model as M

# ---- one output folder for everything --------------------------------------------------------
WORKDIR = OUT / "work"
WORKDIR.mkdir(parents=True, exist_ok=True)
os.environ["TMPDIR"] = str(DIRS["tmp"])
print("all FLAML pilot outputs go to:", OUT)

LEARNERS = {"lgbm": M.LGBMEstimator, "xgboost": M.XGBoostSklearnEstimator,
            "xgb_limitdepth": M.XGBoostLimitDepthEstimator, "rf": M.RandomForestEstimator,
            "extra_tree": M.ExtraTreesEstimator, "catboost": M.CatBoostEstimator,
            "lrl1": M.LRL1Classifier, "lrl2": M.LRL2Classifier, "kneighbor": M.KNeighborsEstimator}
BALANCING = ["none", "balanced"]
JOB_TIME_S, JOB_MEM_MB = 3600, 65536          # 1 h per job; 64 GB virtual memory (CatBoost threads)
N_THREADS = 120                               # jobs running at once

# ---- FLAML task object (rf / extra_tree need it instead of the string "binary") -----------
from flaml.automl.task.factory import task_factory
try:
    TASK = task_factory("binary")
except Exception:
    TASK = task_factory("classification")

def to_py(v):
    return v.item() if isinstance(v, np.generic) else v

def sample_configs(did, learner, k):
    """k unique random configurations from FLAML's own search space for this learner."""
    n, d = len(ROWS[did]["train_idx"]), DATA[did].d
    space = LEARNERS[learner].search_space(data_size=(n, d), task=TASK)
    rng = np.random.RandomState(derive_seed("flaml-hp", did, learner))
    out, seen = [], set()
    for _ in range(50 * k):
        cfg = {hp: (to_py(spec["domain"].sample(random_state=rng)) if hasattr(spec.get("domain"), "sample")
                    else to_py(spec.get("domain"))) for hp, spec in space.items()}
        key = json.dumps(cfg, sort_keys=True, default=str)
        if key not in seen:
            seen.add(key)
            out.append(cfg)
        if len(out) == k:
            break
    return out

# ---- design: learner x configuration x balancing (configuration i shared by both levels) ------
DESIGN = []
for did in CFG.dataset_ids:
    for learner in LEARNERS:
        for i, cfg in enumerate(sample_configs(did, learner, CFG.k_configs)):
            for bal in BALANCING:
                DESIGN.append(dict(did=did, learner=learner, balancing=bal, cfg_idx=i,
                                   pipe_id=f"{learner}|{bal}|{i}", config=cfg))
DESIGN = pd.DataFrame(DESIGN)
DESIGN.drop(columns="config").to_csv(DIRS["data"] / "flaml_design.csv", index=False)
with open(DIRS["data"] / "flaml_configs.json", "w") as f:
    json.dump({f"{r.did}|{r.pipe_id}": r.config for r in DESIGN.itertuples()}, f, default=str, indent=1)
print("pipelines per learner x dataset:")
print(DESIGN.pivot_table(index="learner", columns="did", values="pipe_id", aggfunc="count").to_string())

# ---- worker module (each job runs in its own process; written into results/work/) ------------
JOB_SRC = r'''
import os, sys, json, pickle, time, warnings, resource
import numpy as np, pandas as pd

def _vmsize():
    with open("/proc/self/status") as f:
        for line in f:
            if line.startswith("VmSize:"):
                return int(line.split()[1]) * 1024

def job(a):
    warnings.filterwarnings("ignore")
    soft, hard = resource.getrlimit(resource.RLIMIT_AS)
    limit = _vmsize() + a["mem_limit_mb"] * 1024 ** 2
    resource.setrlimit(resource.RLIMIT_AS, (limit if hard == resource.RLIM_INFINITY else min(limit, hard), hard))
    out = dict(status="ok", err="")
    try:
        from flaml import AutoML
        import shap

        def frame(X):
            df = pd.DataFrame(np.asarray(X, dtype=float), columns=a["feature_names"])
            for j, c in enumerate(a["feature_names"]):
                if a["cat_mask"][j]:
                    df[c] = pd.Categorical(df[c].astype(int), categories=list(range(a["n_levels"][j])))
            return df

        def fit(Xn, yn, seed):
            kw = dict(X_train=frame(Xn), y_train=yn, task="classification", metric="log_loss",
                      estimator_list=[a["learner"]], starting_points={a["learner"]: a["config"]},
                      max_iter=1, retrain_full=True, seed=seed, n_jobs=1, verbose=0)
            if a["balancing"] == "balanced":
                w = len(yn) / (2.0 * np.bincount(yn, minlength=2))
                kw["sample_weight"] = w[yn]
            aml = AutoML()
            aml.fit(**kw)
            return aml

        t0 = time.time()
        aml = fit(a["X_train"], a["y_train"], a["seed"])
        out["fit_secs"] = time.time() - t0
        out["config_match"] = bool(all(aml.best_config.get(k) == v for k, v in a["config"].items()))

        def g(X):
            X = np.asarray(X, dtype=float)
            return np.concatenate([aml.predict_proba(frame(X[i:i + 4096]))[:, 1]
                                   for i in range(0, len(X), 4096)])

        masker = shap.maskers.Independent(a["B"], max_samples=len(a["B"]))
        np.random.seed(a["perm_seed"])
        ex = shap.PermutationExplainer(g, masker, seed=a["perm_seed"])
        np.random.seed(a["perm_seed"])
        t1 = time.time()
        e = ex(a["XE"], max_evals=a["max_evals"], silent=True)
        out["explain_secs"] = time.time() - t1
        phi = np.asarray(e.values)
        base = np.asarray(e.base_values).reshape(-1)
        pE = g(a["XE"])
        out.update(m=np.abs(phi).mean(0), pE=pE,
                   efficiency_error=float(np.abs(phi.sum(1) + base - pE).max()))

        if a["do_cv"]:                                        # 5-fold CV on the fixed Phase-2 folds
            oof = np.empty(len(a["y_train"]))
            for fit_pos, val_pos in a["cv_folds"]:
                m_ = fit(a["X_train"][fit_pos], a["y_train"][fit_pos], a["seed"])
                oof[val_pos] = m_.predict_proba(frame(a["X_train"][val_pos]))[:, 1]
            out["oof"] = oof
    except MemoryError:
        out.update(status="memout")
    except Exception as ex_:
        out.update(status="error", err=f"{type(ex_).__name__}: {str(ex_)[:300]}")
    tmp = a["out_path"] + ".tmp.npz"
    np.savez(tmp, **out)
    os.replace(tmp, a["out_path"])

def run_isolated(a, time_limit_s, workdir):
    import subprocess, tempfile
    fd, apath = tempfile.mkstemp(suffix=".pkl", dir=a["tmp_dir"])
    os.close(fd)
    with open(apath, "wb") as f:
        pickle.dump(a, f, protocol=4)
    t0, status, err = time.time(), None, ""
    try:
        r = subprocess.run([sys.executable, "-m", "flaml_job", apath], cwd=workdir, timeout=time_limit_s,
                           stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
        if r.returncode == -2:                                # interrupted (SIGINT): save nothing, rerun later
            return "interrupted"
        if r.returncode != 0:
            status, err = "crashed", f"exit {r.returncode}; " + r.stderr.decode(errors="ignore")[-300:]
    except subprocess.TimeoutExpired:
        status, err = "timeout", f"exceeded {time_limit_s}s"
    finally:
        if os.path.exists(apath):
            os.remove(apath)
    if os.path.exists(a["out_path"]):
        return "done"
    np.savez(a["out_path"], status=status or "crashed", err=err, total_secs=time.time() - t0)
    return status or "crashed"

if __name__ == "__main__":
    with open(sys.argv[1], "rb") as f:
        job(pickle.load(f))
'''
(WORKDIR / "flaml_job.py").write_text(JOB_SRC)
if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))
import flaml_job
importlib.reload(flaml_job)

# ---- paths and arguments -------------------------------------------------------------------------
def job_path(did, pipe_id, s):
    d = DIRS["xai"] / str(did)
    d.mkdir(exist_ok=True)
    return d / f"{pipe_id.replace('|', '__')}_s{s}.npz"

def job_args(r_, s):
    D, rs = DATA[r_.did], ROWS[r_.did]
    tr = rs["train_idx"]
    pos = {row: k for k, row in enumerate(tr)}
    folds = [(np.array([pos[x] for x in f_]), np.array([pos[x] for x in v_]))
             for f_, v_ in zip(rs["cv_fit"], rs["cv_val"])]
    return dict(learner=r_.learner, config=r_.config, balancing=r_.balancing,
                seed=derive_seed("flaml-train", r_.did, r_.pipe_id, s),
                X_train=D.X[tr], y_train=D.y[tr], feature_names=D.feature_names,
                cat_mask=list(map(bool, D.cat_mask)),
                n_levels=[int(D.X[:, j].max()) + 1 for j in range(D.d)],
                B=D.X[rs["bg_idx"][0]], XE=D.X[rs["E_idx"]], max_evals=CFG.max_evals(D.d),
                perm_seed=derive_seed("perm", r_.did, 0), do_cv=(s == 0), cv_folds=folds,
                mem_limit_mb=JOB_MEM_MB, out_path=str(job_path(r_.did, r_.pipe_id, s)),
                tmp_dir=str(DIRS["tmp"]))

# ---- remove results that must be redone (keep 'ok' and genuine 'error' holes) ----------------
removed = 0
for p in DIRS["xai"].glob("*/*.npz"):
    try:
        st = str(np.load(p)["status"])
    except Exception:
        st = "unreadable"
    if st in ("crashed", "timeout", "memout", "unreadable"):
        p.unlink()
        removed += 1
for p in DIRS["tmp"].glob("*"):
    if p.is_file():
        p.unlink()
print(f"\nremoved {removed} crashed / timed-out / memout results from earlier runs (they will be redone)")

# ---- run: every pipeline x 3 seeds; CV with seed 0 ------------------------------------------
jobs = [(r_, s) for r_ in DESIGN.itertuples() for s in range(CFG.n_train_seeds)
        if not job_path(r_.did, r_.pipe_id, s).exists()]
n_thr = min(N_THREADS, max(1, len(jobs)))
print(f"{len(DESIGN) * CFG.n_train_seeds} jobs in total, {len(jobs)} to run on {n_thr} threads", flush=True)
t0 = time.time()
if jobs:
    res = Parallel(n_jobs=n_thr, backend="threading", verbose=5)(
        delayed(flaml_job.run_isolated)(job_args(r_, s), JOB_TIME_S, str(WORKDIR)) for r_, s in jobs)
    print("run outcomes:", pd.Series(res).value_counts().to_dict())
print(f"finished in {(time.time() - t0) / 60:.1f} min")

# ---- status and checks -----------------------------------------------------------------------
rows = []
for r_ in DESIGN.itertuples():
    for s in range(CFG.n_train_seeds):
        p = job_path(r_.did, r_.pipe_id, s)
        if not p.exists():
            rows.append(dict(did=r_.did, learner=r_.learner, balancing=r_.balancing, seed=s,
                             status="missing (interrupted)", err=""))
            continue
        z = np.load(p)
        st = str(z["status"])
        rows.append(dict(did=r_.did, learner=r_.learner, balancing=r_.balancing, seed=s, status=st,
                         config_match=bool(z["config_match"]) if st == "ok" else np.nan,
                         eff=float(z["efficiency_error"]) if st == "ok" else np.nan,
                         secs=float(z["fit_secs"]) + float(z["explain_secs"]) if st == "ok" else np.nan,
                         err=str(z["err"])[:120] if st != "ok" else ""))
FST = pd.DataFrame(rows)
FST.to_csv(DIRS["metrics"] / "flaml_status.csv", index=False)

print("\nstatus per learner:")
print(FST.pivot_table(index="learner", columns="status", values="seed", aggfunc="count", fill_value=0).to_string())
ok = FST[FST.status == "ok"]
print(f"\nconfiguration used == requested: {int(ok.config_match.sum())} of {len(ok)}")
print(f"max efficiency error: {ok.eff.max():.1e}")
print("\nmedian seconds per job (fit + explain), learner x dataset:")
print(ok.pivot_table(index="learner", columns="did", values="secs", aggfunc="median").round(1).to_string())
bad = FST[~FST.status.isin(["ok"])]
if len(bad):
    print("\nnot ok (kNN + balanced = structural hole: FLAML's kNN has no sample weights):")
    print(bad.groupby(["did", "learner", "balancing", "status"]).size().to_string())
    print(bad.drop_duplicates(["learner", "status"])[["learner", "status", "err"]].to_string())
print("\nfiles written under:", OUT)

all FLAML pilot outputs go to: /ds-dsa/dbanerjee/automl_flaml/results
pipelines per learner x dataset:
did             31  37
learner               
catboost        10  10
extra_tree      10  10
kneighbor       10  10
lgbm            10  10
lrl1            10  10
lrl2            10  10
rf              10  10
xgb_limitdepth  10  10
xgboost         10  10

removed 0 crashed / timed-out / memout results from earlier runs (they will be redone)
540 jobs in total, 187 to run on 120 threads


[Parallel(n_jobs=120)]: Using backend ThreadingBackend with 120 concurrent workers.
[Parallel(n_jobs=120)]: Done  24 out of 187 | elapsed:   21.1s remaining:  2.4min
[Parallel(n_jobs=120)]: Done  62 out of 187 | elapsed:   36.7s remaining:  1.2min
[Parallel(n_jobs=120)]: Done 100 out of 187 | elapsed:   44.6s remaining:   38.8s
[Parallel(n_jobs=120)]: Done 138 out of 187 | elapsed:  1.0min remaining:   22.1s
[Parallel(n_jobs=120)]: Done 176 out of 187 | elapsed:  2.0min remaining:    7.3s
[Parallel(n_jobs=120)]: Done 187 out of 187 | elapsed:  2.3min finished


run outcomes: {'done': 187}
finished in 2.3 min

status per learner:
status          error  ok
learner                  
catboost            0  60
extra_tree          0  60
kneighbor          30  30
lgbm                0  60
lrl1                0  60
lrl2                0  60
rf                  0  60
xgb_limitdepth      0  60
xgboost             0  60

configuration used == requested: 510 of 510
max efficiency error: 2.7e-15

median seconds per job (fit + explain), learner x dataset:
did               31    37
learner                   
catboost        76.9  21.6
extra_tree      77.0  18.7
kneighbor       60.3  31.3
lgbm            77.3  31.0
lrl1            57.0  14.6
lrl2            58.0  13.4
rf              75.1  17.4
xgb_limitdepth  78.3  32.8
xgboost         83.6  37.5

not ok (kNN + balanced = structural hole: FLAML's kNN has no sample weights):
did  learner    balancing  status
31   kneighbor  balanced   error     15
37   kneighbor  balanced   error     15
       learner statu

In [6]:
# =============================================================================
# CELL F2 (v3) — FLAML pilot: design, train with 3 REAL seeds, explain like Phase 2,
#   5-fold CV (seed 0). Fix: FLAML's AutoML(seed=...) does not reach the models, so
#   each learner's own random state is fixed via custom_hp. Old results are reset.
#   ALL outputs go to /ds-dsa/dbanerjee/automl_flaml/results/
# =============================================================================
import pickle, importlib, shutil
from joblib import Parallel, delayed
from flaml.automl import model as M

RESET = True                                  # delete previous FLAML job results (models had no real seeds)

WORKDIR = OUT / "work"
WORKDIR.mkdir(parents=True, exist_ok=True)
os.environ["TMPDIR"] = str(DIRS["tmp"])
print("all FLAML pilot outputs go to:", OUT)

LEARNERS = {"lgbm": M.LGBMEstimator, "xgboost": M.XGBoostSklearnEstimator,
            "xgb_limitdepth": M.XGBoostLimitDepthEstimator, "rf": M.RandomForestEstimator,
            "extra_tree": M.ExtraTreesEstimator, "catboost": M.CatBoostEstimator,
            "lrl1": M.LRL1Classifier, "lrl2": M.LRL2Classifier, "kneighbor": M.KNeighborsEstimator}
SEED_KEY = {"lgbm": "random_state", "xgboost": "random_state", "xgb_limitdepth": "random_state",
            "rf": "random_state", "extra_tree": "random_state", "catboost": "random_seed",
            "lrl1": "random_state", "lrl2": "random_state", "kneighbor": None}
BALANCING = ["none", "balanced"]
JOB_TIME_S, JOB_MEM_MB = 3600, 65536
N_THREADS = 120

from flaml.automl.task.factory import task_factory
try:
    TASK = task_factory("binary")
except Exception:
    TASK = task_factory("classification")

def to_py(v):
    return v.item() if isinstance(v, np.generic) else v

def sample_configs(did, learner, k):
    n, d = len(ROWS[did]["train_idx"]), DATA[did].d
    space = LEARNERS[learner].search_space(data_size=(n, d), task=TASK)
    rng = np.random.RandomState(derive_seed("flaml-hp", did, learner))
    out, seen = [], set()
    for _ in range(50 * k):
        cfg = {hp: (to_py(spec["domain"].sample(random_state=rng)) if hasattr(spec.get("domain"), "sample")
                    else to_py(spec.get("domain"))) for hp, spec in space.items()}
        key = json.dumps(cfg, sort_keys=True, default=str)
        if key not in seen:
            seen.add(key)
            out.append(cfg)
        if len(out) == k:
            break
    return out

DESIGN = []
for did in CFG.dataset_ids:
    for learner in LEARNERS:
        for i, cfg in enumerate(sample_configs(did, learner, CFG.k_configs)):
            for bal in BALANCING:
                DESIGN.append(dict(did=did, learner=learner, balancing=bal, cfg_idx=i,
                                   pipe_id=f"{learner}|{bal}|{i}", config=cfg))
DESIGN = pd.DataFrame(DESIGN)
DESIGN.drop(columns="config").to_csv(DIRS["data"] / "flaml_design.csv", index=False)
with open(DIRS["data"] / "flaml_configs.json", "w") as f:
    json.dump({f"{r.did}|{r.pipe_id}": r.config for r in DESIGN.itertuples()}, f, default=str, indent=1)
print("pipelines per learner x dataset:")
print(DESIGN.pivot_table(index="learner", columns="did", values="pipe_id", aggfunc="count").to_string())

JOB_SRC = r'''
import os, sys, json, pickle, time, warnings, resource
import numpy as np, pandas as pd

def _vmsize():
    with open("/proc/self/status") as f:
        for line in f:
            if line.startswith("VmSize:"):
                return int(line.split()[1]) * 1024

def job(a):
    warnings.filterwarnings("ignore")
    soft, hard = resource.getrlimit(resource.RLIMIT_AS)
    limit = _vmsize() + a["mem_limit_mb"] * 1024 ** 2
    resource.setrlimit(resource.RLIMIT_AS, (limit if hard == resource.RLIM_INFINITY else min(limit, hard), hard))
    out = dict(status="ok", err="")
    try:
        from flaml import AutoML
        import shap

        def frame(X):
            df = pd.DataFrame(np.asarray(X, dtype=float), columns=a["feature_names"])
            for j, c in enumerate(a["feature_names"]):
                if a["cat_mask"][j]:
                    df[c] = pd.Categorical(df[c].astype(int), categories=list(range(a["n_levels"][j])))
            return df

        def fit(Xn, yn, seed):
            kw = dict(X_train=frame(Xn), y_train=yn, task="classification", metric="log_loss",
                      estimator_list=[a["learner"]], starting_points={a["learner"]: a["config"]},
                      max_iter=1, retrain_full=True, seed=seed, n_jobs=1, verbose=0)
            if a["seed_key"]:                                  # give the MODEL its own seed
                kw["custom_hp"] = {a["learner"]: {a["seed_key"]: {"domain": int(seed)}}}
            if a["balancing"] == "balanced":
                w = len(yn) / (2.0 * np.bincount(yn, minlength=2))
                kw["sample_weight"] = w[yn]
            aml = AutoML()
            aml.fit(**kw)
            return aml

        t0 = time.time()
        aml = fit(a["X_train"], a["y_train"], a["seed"])
        out["fit_secs"] = time.time() - t0
        out["config_match"] = bool(all(aml.best_config.get(k) == v for k, v in a["config"].items()))
        out["seed_used"] = str(aml.best_config.get(a["seed_key"])) if a["seed_key"] else "n/a"

        def g(X):
            X = np.asarray(X, dtype=float)
            return np.concatenate([aml.predict_proba(frame(X[i:i + 4096]))[:, 1]
                                   for i in range(0, len(X), 4096)])

        masker = shap.maskers.Independent(a["B"], max_samples=len(a["B"]))
        np.random.seed(a["perm_seed"])
        ex = shap.PermutationExplainer(g, masker, seed=a["perm_seed"])
        np.random.seed(a["perm_seed"])
        t1 = time.time()
        e = ex(a["XE"], max_evals=a["max_evals"], silent=True)
        out["explain_secs"] = time.time() - t1
        phi = np.asarray(e.values)
        base = np.asarray(e.base_values).reshape(-1)
        pE = g(a["XE"])
        out.update(m=np.abs(phi).mean(0), pE=pE,
                   efficiency_error=float(np.abs(phi.sum(1) + base - pE).max()))

        if a["do_cv"]:
            oof = np.empty(len(a["y_train"]))
            for fit_pos, val_pos in a["cv_folds"]:
                m_ = fit(a["X_train"][fit_pos], a["y_train"][fit_pos], a["seed"])
                oof[val_pos] = m_.predict_proba(frame(a["X_train"][val_pos]))[:, 1]
            out["oof"] = oof
    except MemoryError:
        out.update(status="memout")
    except Exception as ex_:
        out.update(status="error", err=f"{type(ex_).__name__}: {str(ex_)[:300]}")
    tmp = a["out_path"] + ".tmp.npz"
    np.savez(tmp, **out)
    os.replace(tmp, a["out_path"])

def run_isolated(a, time_limit_s, workdir):
    import subprocess, tempfile
    fd, apath = tempfile.mkstemp(suffix=".pkl", dir=a["tmp_dir"])
    os.close(fd)
    with open(apath, "wb") as f:
        pickle.dump(a, f, protocol=4)
    t0, status, err = time.time(), None, ""
    try:
        r = subprocess.run([sys.executable, "-m", "flaml_job", apath], cwd=workdir, timeout=time_limit_s,
                           stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
        if r.returncode == -2:
            return "interrupted"
        if r.returncode != 0:
            status, err = "crashed", f"exit {r.returncode}; " + r.stderr.decode(errors="ignore")[-300:]
    except subprocess.TimeoutExpired:
        status, err = "timeout", f"exceeded {time_limit_s}s"
    finally:
        if os.path.exists(apath):
            os.remove(apath)
    if os.path.exists(a["out_path"]):
        return "done"
    np.savez(a["out_path"], status=status or "crashed", err=err, total_secs=time.time() - t0)
    return status or "crashed"

if __name__ == "__main__":
    with open(sys.argv[1], "rb") as f:
        job(pickle.load(f))
'''
(WORKDIR / "flaml_job.py").write_text(JOB_SRC)
if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))
import flaml_job
importlib.reload(flaml_job)

def job_path(did, pipe_id, s):
    d = DIRS["xai"] / str(did)
    d.mkdir(exist_ok=True)
    return d / f"{pipe_id.replace('|', '__')}_s{s}.npz"

def job_args(r_, s):
    D, rs = DATA[r_.did], ROWS[r_.did]
    tr = rs["train_idx"]
    pos = {row: k for k, row in enumerate(tr)}
    folds = [(np.array([pos[x] for x in f_]), np.array([pos[x] for x in v_]))
             for f_, v_ in zip(rs["cv_fit"], rs["cv_val"])]
    return dict(learner=r_.learner, config=r_.config, balancing=r_.balancing, seed_key=SEED_KEY[r_.learner],
                seed=derive_seed("flaml-train", r_.did, r_.pipe_id, s),
                X_train=D.X[tr], y_train=D.y[tr], feature_names=D.feature_names,
                cat_mask=list(map(bool, D.cat_mask)),
                n_levels=[int(D.X[:, j].max()) + 1 for j in range(D.d)],
                B=D.X[rs["bg_idx"][0]], XE=D.X[rs["E_idx"]], max_evals=CFG.max_evals(D.d),
                perm_seed=derive_seed("perm", r_.did, 0), do_cv=(s == 0), cv_folds=folds,
                mem_limit_mb=JOB_MEM_MB, out_path=str(job_path(r_.did, r_.pipe_id, s)),
                tmp_dir=str(DIRS["tmp"]))

# ---- reset old results (models had no real seeds) ------------------------------------------------
if RESET:
    n_old = len(list(DIRS["xai"].glob("*/*.npz")))
    shutil.rmtree(DIRS["xai"], ignore_errors=True)
    DIRS["xai"].mkdir(parents=True, exist_ok=True)
    print(f"\nRESET: removed {n_old} old FLAML job results")
for p in DIRS["tmp"].glob("*"):
    if p.is_file():
        p.unlink()

jobs = [(r_, s) for r_ in DESIGN.itertuples() for s in range(CFG.n_train_seeds)
        if not job_path(r_.did, r_.pipe_id, s).exists()]
n_thr = min(N_THREADS, max(1, len(jobs)))
print(f"{len(DESIGN) * CFG.n_train_seeds} jobs in total, {len(jobs)} to run on {n_thr} threads", flush=True)
t0 = time.time()
if jobs:
    res = Parallel(n_jobs=n_thr, backend="threading", verbose=5)(
        delayed(flaml_job.run_isolated)(job_args(r_, s), JOB_TIME_S, str(WORKDIR)) for r_, s in jobs)
    print("run outcomes:", pd.Series(res).value_counts().to_dict())
print(f"finished in {(time.time() - t0) / 60:.1f} min")

# ---- status and checks --------------------------------------------------------------------------
rows = []
for r_ in DESIGN.itertuples():
    zs = []
    for s in range(CFG.n_train_seeds):
        p = job_path(r_.did, r_.pipe_id, s)
        if not p.exists():
            rows.append(dict(did=r_.did, learner=r_.learner, balancing=r_.balancing, seed=s, status="missing")); continue
        z = np.load(p); zs.append(z)
        st = str(z["status"])
        rows.append(dict(did=r_.did, learner=r_.learner, balancing=r_.balancing, seed=s, status=st,
                         config_match=bool(z["config_match"]) if st == "ok" else np.nan,
                         eff=float(z["efficiency_error"]) if st == "ok" else np.nan,
                         err=str(z["err"])[:120] if st != "ok" else ""))
FST = pd.DataFrame(rows)
FST.to_csv(DIRS["metrics"] / "flaml_status.csv", index=False)
print("\nstatus per learner:")
print(FST.pivot_table(index="learner", columns="status", values="seed", aggfunc="count", fill_value=0).to_string())
ok = FST[FST.status == "ok"]
print(f"\nconfiguration used == requested: {int(ok.config_match.sum())} of {len(ok)}")
print(f"max efficiency error: {ok.eff.max():.1e}")

chk = []
for r_ in DESIGN.itertuples():
    zs = [np.load(job_path(r_.did, r_.pipe_id, s)) for s in range(CFG.n_train_seeds)]
    if all(str(z["status"]) == "ok" for z in zs):
        chk.append(dict(did=r_.did, learner=r_.learner,
                        seeds_identical=all(np.array_equal(zs[0]["pE"], z["pE"]) for z in zs[1:])))
print("\nSeed check after the fix (1.00 = seeds have no effect; expected ~0 for random learners, 1 for kNN):")
print(pd.DataFrame(chk).pivot_table(index="learner", columns="did", values="seeds_identical", aggfunc="mean")
      .round(2).to_string())
bad = FST[FST.status != "ok"]
if len(bad):
    print("\nnot ok:")
    print(bad.groupby(["did", "learner", "balancing", "status"]).size().to_string())
    print(bad.drop_duplicates(["learner", "status"])[["learner", "status", "err"]].to_string())

all FLAML pilot outputs go to: /ds-dsa/dbanerjee/automl_flaml/results
pipelines per learner x dataset:
did             31  37
learner               
catboost        10  10
extra_tree      10  10
kneighbor       10  10
lgbm            10  10
lrl1            10  10
lrl2            10  10
rf              10  10
xgb_limitdepth  10  10
xgboost         10  10

RESET: removed 540 old FLAML job results
540 jobs in total, 540 to run on 120 threads


[Parallel(n_jobs=120)]: Using backend ThreadingBackend with 120 concurrent workers.
[Parallel(n_jobs=120)]: Done  48 tasks      | elapsed:   36.4s
[Parallel(n_jobs=120)]: Done 210 tasks      | elapsed:  1.2min
[Parallel(n_jobs=120)]: Done 410 out of 540 | elapsed:  3.7min remaining:  1.2min
[Parallel(n_jobs=120)]: Done 519 out of 540 | elapsed:  4.4min remaining:   10.7s
[Parallel(n_jobs=120)]: Done 540 out of 540 | elapsed:  6.5min finished


run outcomes: {'done': 540}
finished in 6.5 min

status per learner:
status          error  ok
learner                  
catboost            0  60
extra_tree          0  60
kneighbor          30  30
lgbm                0  60
lrl1                0  60
lrl2                0  60
rf                  0  60
xgb_limitdepth      0  60
xgboost             0  60

configuration used == requested: 510 of 510
max efficiency error: 2.7e-15

Seed check after the fix (1.00 = seeds have no effect; expected ~0 for random learners, 1 for kNN):
did              31   37
learner                 
catboost        1.0  1.0
extra_tree      1.0  1.0
kneighbor       1.0  1.0
lgbm            1.0  1.0
lrl1            1.0  1.0
lrl2            1.0  1.0
rf              1.0  1.0
xgb_limitdepth  1.0  1.0
xgboost         1.0  1.0

not ok:
did  learner    balancing  status
31   kneighbor  balanced   error     15
37   kneighbor  balanced   error     15
       learner status                                                  

In [7]:
# =============================================================================
# CELL F2 (v4) — FLAML pilot with REAL training seeds.
#   The model seed is enforced three ways: (1) in the starting configuration,
#   (2) via custom_hp, (3) after fitting: if the model's random state differs,
#   it is set in FLAML's model params and the model is refit on FLAML's own
#   preprocessed data. The seed actually inside each model is recorded.
#   ALL outputs go to /ds-dsa/dbanerjee/automl_flaml/results/
# =============================================================================
import pickle, importlib, shutil
from joblib import Parallel, delayed
from flaml.automl import model as M

RESET = True                                  # delete previous FLAML job results

WORKDIR = OUT / "work"
WORKDIR.mkdir(parents=True, exist_ok=True)
os.environ["TMPDIR"] = str(DIRS["tmp"])
print("all FLAML pilot outputs go to:", OUT)

LEARNERS = {"lgbm": M.LGBMEstimator, "xgboost": M.XGBoostSklearnEstimator,
            "xgb_limitdepth": M.XGBoostLimitDepthEstimator, "rf": M.RandomForestEstimator,
            "extra_tree": M.ExtraTreesEstimator, "catboost": M.CatBoostEstimator,
            "lrl1": M.LRL1Classifier, "lrl2": M.LRL2Classifier, "kneighbor": M.KNeighborsEstimator}
SEED_KEY = {"lgbm": "random_state", "xgboost": "random_state", "xgb_limitdepth": "random_state",
            "rf": "random_state", "extra_tree": "random_state", "catboost": "random_seed",
            "lrl1": "random_state", "lrl2": "random_state", "kneighbor": None}
BALANCING = ["none", "balanced"]
JOB_TIME_S, JOB_MEM_MB = 3600, 65536
N_THREADS = 120

from flaml.automl.task.factory import task_factory
try:
    TASK = task_factory("binary")
except Exception:
    TASK = task_factory("classification")

def to_py(v):
    return v.item() if isinstance(v, np.generic) else v

def sample_configs(did, learner, k):
    n, d = len(ROWS[did]["train_idx"]), DATA[did].d
    space = LEARNERS[learner].search_space(data_size=(n, d), task=TASK)
    rng = np.random.RandomState(derive_seed("flaml-hp", did, learner))
    out, seen = [], set()
    for _ in range(50 * k):
        cfg = {hp: (to_py(spec["domain"].sample(random_state=rng)) if hasattr(spec.get("domain"), "sample")
                    else to_py(spec.get("domain"))) for hp, spec in space.items()}
        key = json.dumps(cfg, sort_keys=True, default=str)
        if key not in seen:
            seen.add(key)
            out.append(cfg)
        if len(out) == k:
            break
    return out

DESIGN = []
for did in CFG.dataset_ids:
    for learner in LEARNERS:
        for i, cfg in enumerate(sample_configs(did, learner, CFG.k_configs)):
            for bal in BALANCING:
                DESIGN.append(dict(did=did, learner=learner, balancing=bal, cfg_idx=i,
                                   pipe_id=f"{learner}|{bal}|{i}", config=cfg))
DESIGN = pd.DataFrame(DESIGN)
DESIGN.drop(columns="config").to_csv(DIRS["data"] / "flaml_design.csv", index=False)
with open(DIRS["data"] / "flaml_configs.json", "w") as f:
    json.dump({f"{r.did}|{r.pipe_id}": r.config for r in DESIGN.itertuples()}, f, default=str, indent=1)
print("pipelines per learner x dataset:")
print(DESIGN.pivot_table(index="learner", columns="did", values="pipe_id", aggfunc="count").to_string())

JOB_SRC = r'''
import os, sys, json, pickle, time, warnings, resource
import numpy as np, pandas as pd

def _vmsize():
    with open("/proc/self/status") as f:
        for line in f:
            if line.startswith("VmSize:"):
                return int(line.split()[1]) * 1024

def model_seed(aml, key):
    """Random state actually inside the trained model."""
    try:
        est = aml.model.estimator
        params = est.get_params() if hasattr(est, "get_params") else {}
        if key in params:
            return params[key]
        if hasattr(est, "get_param"):                       # CatBoost
            return est.get_param(key)
    except Exception:
        pass
    return None

def job(a):
    warnings.filterwarnings("ignore")
    soft, hard = resource.getrlimit(resource.RLIMIT_AS)
    limit = _vmsize() + a["mem_limit_mb"] * 1024 ** 2
    resource.setrlimit(resource.RLIMIT_AS, (limit if hard == resource.RLIM_INFINITY else min(limit, hard), hard))
    out = dict(status="ok", err="", seed_route="n/a")
    try:
        from flaml import AutoML
        import shap

        def frame(X):
            df = pd.DataFrame(np.asarray(X, dtype=float), columns=a["feature_names"])
            for j, c in enumerate(a["feature_names"]):
                if a["cat_mask"][j]:
                    df[c] = pd.Categorical(df[c].astype(int), categories=list(range(a["n_levels"][j])))
            return df

        def fit(Xn, yn, seed):
            key = a["seed_key"]
            start = dict(a["config"])
            if key:
                start[key] = int(seed)                          # (1) seed in the starting configuration
            kw = dict(X_train=frame(Xn), y_train=yn, task="classification", metric="log_loss",
                      estimator_list=[a["learner"]], starting_points={a["learner"]: start},
                      max_iter=1, retrain_full=True, seed=seed, n_jobs=1, verbose=0)
            if key:
                kw["custom_hp"] = {a["learner"]: {key: {"domain": int(seed)}}}   # (2) custom_hp
            w = None
            if a["balancing"] == "balanced":
                cw = len(yn) / (2.0 * np.bincount(yn, minlength=2))
                w = cw[yn]
                kw["sample_weight"] = w
            aml = AutoML()
            aml.fit(**kw)
            route = "config/custom_hp"
            if key and model_seed(aml, key) != int(seed):       # (3) enforce after fitting
                route = "refit"
                aml.model.params[key] = int(seed)
                Xt = aml._transformer.transform(frame(Xn)) if getattr(aml, "_transformer", None) else frame(Xn)
                yt = aml._label_transformer.transform(yn) if getattr(aml, "_label_transformer", None) else yn
                fit_kw = {"sample_weight": w} if w is not None else {}
                aml.model.fit(Xt, yt, **fit_kw)
                if model_seed(aml, key) != int(seed):
                    route = "FAILED"
            return aml, route

        t0 = time.time()
        aml, route = fit(a["X_train"], a["y_train"], a["seed"])
        out["fit_secs"] = time.time() - t0
        out["seed_route"] = route
        out["seed_in_model"] = str(model_seed(aml, a["seed_key"])) if a["seed_key"] else "n/a"
        out["config_match"] = bool(all(aml.best_config.get(k) == v for k, v in a["config"].items()))

        def g(X):
            X = np.asarray(X, dtype=float)
            return np.concatenate([aml.predict_proba(frame(X[i:i + 4096]))[:, 1]
                                   for i in range(0, len(X), 4096)])

        masker = shap.maskers.Independent(a["B"], max_samples=len(a["B"]))
        np.random.seed(a["perm_seed"])
        ex = shap.PermutationExplainer(g, masker, seed=a["perm_seed"])
        np.random.seed(a["perm_seed"])
        t1 = time.time()
        e = ex(a["XE"], max_evals=a["max_evals"], silent=True)
        out["explain_secs"] = time.time() - t1
        phi = np.asarray(e.values)
        base = np.asarray(e.base_values).reshape(-1)
        pE = g(a["XE"])
        out.update(m=np.abs(phi).mean(0), pE=pE,
                   efficiency_error=float(np.abs(phi.sum(1) + base - pE).max()))

        if a["do_cv"]:
            oof = np.empty(len(a["y_train"]))
            for fit_pos, val_pos in a["cv_folds"]:
                m_, _ = fit(a["X_train"][fit_pos], a["y_train"][fit_pos], a["seed"])
                oof[val_pos] = m_.predict_proba(frame(a["X_train"][val_pos]))[:, 1]
            out["oof"] = oof
    except MemoryError:
        out.update(status="memout")
    except Exception as ex_:
        out.update(status="error", err=f"{type(ex_).__name__}: {str(ex_)[:300]}")
    tmp = a["out_path"] + ".tmp.npz"
    np.savez(tmp, **out)
    os.replace(tmp, a["out_path"])

def run_isolated(a, time_limit_s, workdir):
    import subprocess, tempfile
    fd, apath = tempfile.mkstemp(suffix=".pkl", dir=a["tmp_dir"])
    os.close(fd)
    with open(apath, "wb") as f:
        pickle.dump(a, f, protocol=4)
    t0, status, err = time.time(), None, ""
    try:
        r = subprocess.run([sys.executable, "-m", "flaml_job", apath], cwd=workdir, timeout=time_limit_s,
                           stdout=subprocess.DEVNULL, stderr=subprocess.PIPE)
        if r.returncode == -2:
            return "interrupted"
        if r.returncode != 0:
            status, err = "crashed", f"exit {r.returncode}; " + r.stderr.decode(errors="ignore")[-300:]
    except subprocess.TimeoutExpired:
        status, err = "timeout", f"exceeded {time_limit_s}s"
    finally:
        if os.path.exists(apath):
            os.remove(apath)
    if os.path.exists(a["out_path"]):
        return "done"
    np.savez(a["out_path"], status=status or "crashed", err=err, total_secs=time.time() - t0)
    return status or "crashed"

if __name__ == "__main__":
    with open(sys.argv[1], "rb") as f:
        job(pickle.load(f))
'''
(WORKDIR / "flaml_job.py").write_text(JOB_SRC)
if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))
import flaml_job
importlib.reload(flaml_job)

def job_path(did, pipe_id, s):
    d = DIRS["xai"] / str(did)
    d.mkdir(exist_ok=True)
    return d / f"{pipe_id.replace('|', '__')}_s{s}.npz"

def job_args(r_, s):
    D, rs = DATA[r_.did], ROWS[r_.did]
    tr = rs["train_idx"]
    pos = {row: k for k, row in enumerate(tr)}
    folds = [(np.array([pos[x] for x in f_]), np.array([pos[x] for x in v_]))
             for f_, v_ in zip(rs["cv_fit"], rs["cv_val"])]
    return dict(learner=r_.learner, config=r_.config, balancing=r_.balancing, seed_key=SEED_KEY[r_.learner],
                seed=derive_seed("flaml-train", r_.did, r_.pipe_id, s),
                X_train=D.X[tr], y_train=D.y[tr], feature_names=D.feature_names,
                cat_mask=list(map(bool, D.cat_mask)),
                n_levels=[int(D.X[:, j].max()) + 1 for j in range(D.d)],
                B=D.X[rs["bg_idx"][0]], XE=D.X[rs["E_idx"]], max_evals=CFG.max_evals(D.d),
                perm_seed=derive_seed("perm", r_.did, 0), do_cv=(s == 0), cv_folds=folds,
                mem_limit_mb=JOB_MEM_MB, out_path=str(job_path(r_.did, r_.pipe_id, s)),
                tmp_dir=str(DIRS["tmp"]))

# ---- quick in-notebook test on one random forest before the full run ----------------------------
test_r = next(DESIGN[(DESIGN.did == 37) & (DESIGN.learner == "rf") & (DESIGN.balancing == "none")].itertuples())
test_out = []
for s in (0, 1):
    a = job_args(test_r, s)
    a["out_path"] = str(DIRS["tmp"] / f"seedtest_s{s}.npz"); a["do_cv"] = False
    flaml_job.run_isolated(a, 600, str(WORKDIR))
    z = np.load(a["out_path"]); test_out.append(z)
    print(f"seed test s{s}: status {z['status']} | route {z['seed_route']} | seed requested {a['seed']} "
          f"| seed in model {z['seed_in_model'] if 'seed_in_model' in z else '-'} | err {str(z['err'])[:120]}")
if all(str(z["status"]) == "ok" for z in test_out):
    same = np.array_equal(test_out[0]["pE"], test_out[1]["pE"])
    print("random forest, two seeds -> identical predictions:", same)
    assert not same, "seeds still do not change the model: stopping before the full run"
    print("seed fix works -> running the full design")

# ---- reset and run -------------------------------------------------------------------------------
if RESET:
    n_old = len(list(DIRS["xai"].glob("*/*.npz")))
    shutil.rmtree(DIRS["xai"], ignore_errors=True)
    DIRS["xai"].mkdir(parents=True, exist_ok=True)
    print(f"\nRESET: removed {n_old} old FLAML job results")
for p in DIRS["tmp"].glob("*"):
    if p.is_file():
        p.unlink()

jobs = [(r_, s) for r_ in DESIGN.itertuples() for s in range(CFG.n_train_seeds)
        if not job_path(r_.did, r_.pipe_id, s).exists()]
n_thr = min(N_THREADS, max(1, len(jobs)))
print(f"{len(DESIGN) * CFG.n_train_seeds} jobs in total, {len(jobs)} to run on {n_thr} threads", flush=True)
t0 = time.time()
if jobs:
    res = Parallel(n_jobs=n_thr, backend="threading", verbose=5)(
        delayed(flaml_job.run_isolated)(job_args(r_, s), JOB_TIME_S, str(WORKDIR)) for r_, s in jobs)
    print("run outcomes:", pd.Series(res).value_counts().to_dict())
print(f"finished in {(time.time() - t0) / 60:.1f} min")

# ---- status and checks --------------------------------------------------------------------------
rows = []
for r_ in DESIGN.itertuples():
    for s in range(CFG.n_train_seeds):
        p = job_path(r_.did, r_.pipe_id, s)
        if not p.exists():
            rows.append(dict(did=r_.did, learner=r_.learner, balancing=r_.balancing, seed=s, status="missing"))
            continue
        z = np.load(p)
        st = str(z["status"])
        rows.append(dict(did=r_.did, learner=r_.learner, balancing=r_.balancing, seed=s, status=st,
                         route=str(z["seed_route"]) if "seed_route" in z else "",
                         config_match=bool(z["config_match"]) if st == "ok" else np.nan,
                         eff=float(z["efficiency_error"]) if st == "ok" else np.nan,
                         err=str(z["err"])[:120] if st != "ok" else ""))
FST = pd.DataFrame(rows)
FST.to_csv(DIRS["metrics"] / "flaml_status.csv", index=False)
print("\nstatus per learner:")
print(FST.pivot_table(index="learner", columns="status", values="seed", aggfunc="count", fill_value=0).to_string())
print("\nhow the seed reached each model (route):")
print(FST[FST.status == "ok"].pivot_table(index="learner", columns="route", values="seed", aggfunc="count", fill_value=0).to_string())
ok = FST[FST.status == "ok"]
print(f"\nconfiguration used == requested: {int(ok.config_match.sum())} of {len(ok)}")
print(f"max efficiency error: {ok.eff.max():.1e}")

chk = []
for r_ in DESIGN.itertuples():
    zs = [np.load(job_path(r_.did, r_.pipe_id, s)) for s in range(CFG.n_train_seeds)]
    if all(str(z["status"]) == "ok" for z in zs):
        chk.append(dict(did=r_.did, learner=r_.learner,
                        seeds_identical=all(np.array_equal(zs[0]["pE"], z["pE"]) for z in zs[1:])))
print("\nSeed check (1.00 = seeds have no effect; random learners should now be ~0, kNN stays 1):")
print(pd.DataFrame(chk).pivot_table(index="learner", columns="did", values="seeds_identical", aggfunc="mean")
      .round(2).to_string())
bad = FST[FST.status != "ok"]
if len(bad):
    print("\nnot ok:")
    print(bad.groupby(["did", "learner", "balancing", "status"]).size().to_string())
    print(bad.drop_duplicates(["learner", "status"])[["learner", "status", "err"]].to_string())

all FLAML pilot outputs go to: /ds-dsa/dbanerjee/automl_flaml/results
pipelines per learner x dataset:
did             31  37
learner               
catboost        10  10
extra_tree      10  10
kneighbor       10  10
lgbm            10  10
lrl1            10  10
lrl2            10  10
rf              10  10
xgb_limitdepth  10  10
xgboost         10  10
seed test s0: status ok | route config/custom_hp | seed requested 1022021515 | seed in model 1022021515 | err 
seed test s1: status ok | route config/custom_hp | seed requested 1361563151 | seed in model 1361563151 | err 
random forest, two seeds -> identical predictions: False
seed fix works -> running the full design

RESET: removed 540 old FLAML job results
540 jobs in total, 540 to run on 120 threads


[Parallel(n_jobs=120)]: Using backend ThreadingBackend with 120 concurrent workers.
[Parallel(n_jobs=120)]: Done  48 tasks      | elapsed:   33.1s
[Parallel(n_jobs=120)]: Done 210 tasks      | elapsed:  1.1min
[Parallel(n_jobs=120)]: Done 410 out of 540 | elapsed:  3.6min remaining:  1.1min
[Parallel(n_jobs=120)]: Done 519 out of 540 | elapsed:  4.4min remaining:   10.7s
[Parallel(n_jobs=120)]: Done 540 out of 540 | elapsed:  6.5min finished


run outcomes: {'done': 540}
finished in 6.5 min

status per learner:
status          error  ok
learner                  
catboost            0  60
extra_tree          0  60
kneighbor          30  30
lgbm                0  60
lrl1                0  60
lrl2                0  60
rf                  0  60
xgb_limitdepth      0  60
xgboost             0  60

how the seed reached each model (route):
route           config/custom_hp
learner                         
catboost                      60
extra_tree                    60
kneighbor                     30
lgbm                          60
lrl1                          60
lrl2                          60
rf                            60
xgb_limitdepth                60
xgboost                       60

configuration used == requested: 510 of 510
max efficiency error: 2.7e-15

Seed check (1.00 = seeds have no effect; random learners should now be ~0, kNN stays 1):
did              31   37
learner                 
catboost        0.0  0.0


In [8]:
# =============================================================================
# CELL F3 — FLAML pilot analysis (same steps as auto-sklearn Phase 2):
#   0) seed check: do different training seeds change FLAML's models?
#   CV log-loss + Rashomon sets, delta_0, D_bar/delta_0, Shapley shares
#   (learner vs balancing), nested PERMANOVA (family vs hyperparameters),
#   paired effects, and a side-by-side comparison with auto-sklearn.
#   Safe when a noise floor is exactly 0 (reported as inf).
# =============================================================================
from itertools import combinations
from math import factorial
from scipy.spatial.distance import cdist
from scipy.linalg import orth
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

TOPK, RBO_P = (3, 5), 0.9
DIST_NAMES = ["tv", "jac3", "jac5", "rbo"]
N_PERM = 9999

def safe_ratio(a, b):
    return (a / b) if b > 0 else (np.inf if a > 0 else np.nan)

# ---------------------------------------------------------------- 0) seed check ----
chk = []
for r_ in DESIGN.itertuples():
    zs = [np.load(job_path(r_.did, r_.pipe_id, s)) for s in range(CFG.n_train_seeds)]
    if any(str(z["status"]) != "ok" for z in zs):
        continue
    chk.append(dict(did=r_.did, learner=r_.learner,
                    seeds_identical=all(np.array_equal(zs[0]["pE"], z["pE"]) for z in zs[1:])))
SEEDCHK = pd.DataFrame(chk)
print("Seed check: share of pipelines whose 3 training seeds give the IDENTICAL model")
print("(1.00 = seed has no effect, 0.00 = seeds change the model)")
print(SEEDCHK.pivot_table(index="learner", columns="did", values="seeds_identical", aggfunc="mean").round(2).to_string())
SEEDCHK.to_csv(DIRS["metrics"] / "flaml_seed_check.csv", index=False)

# ---------------------------------------------------------------- measurement library ----
def normalise(m):
    m = np.asarray(m, float); t = m.sum()
    if not np.isfinite(t) or t <= 0:
        return None
    s = m / t
    order = np.lexsort((np.arange(len(s)), -s))
    r = np.empty(len(s), int); r[order] = np.arange(1, len(s) + 1)
    return s, r

def pairwise_D(S, R, p=RBO_P):
    n, d = S.shape
    D = {"tv": 0.5 * cdist(S, S, metric="cityblock")}
    for k in TOPK:
        T = (R <= k).astype(float); inter = T @ T.T
        D[f"jac{k}"] = 1.0 - inter / (2 * k - inter)
    acc = np.zeros((n, n))
    for depth in range(1, d + 1):
        I = (R <= depth).astype(float); acc += (I @ I.T / depth) * p ** depth
    D["rbo"] = 1.0 - (p ** d + ((1 - p) / p) * acc)
    for k in D:
        np.fill_diagonal(D[k], 0.0); np.clip(D[k], 0, 1, out=D[k])
    return D

def mean_pairwise(Dm, idx=None):
    if idx is not None:
        Dm = Dm[np.ix_(idx, idx)]
    iu = np.triu_indices(len(Dm), 1)
    return float(Dm[iu].mean()) if len(iu[0]) else np.nan

def gower(D):
    A = -0.5 * np.asarray(D, float) ** 2
    return A - A.mean(0, keepdims=True) - A.mean(1, keepdims=True) + A.mean()

def dummies(v):
    codes, uniq = pd.factorize(pd.Series(v).astype(str), sort=True)
    Mx = np.zeros((len(codes), len(uniq))); Mx[np.arange(len(codes)), codes] = 1
    return Mx[:, 1:]

def hat(X):
    Q = orth(X); return Q @ Q.T

def r2(G, trG, blocks):
    n = len(G)
    X = np.hstack([np.ones((n, 1))] + [b for b in blocks if b.shape[1]])
    return float(np.vdot(hat(X), G)) / trG

def shapley(D, players):
    """players: {name: [design blocks]} -> (Shapley shares of R^2, total R^2), or None if no dispersion."""
    G = gower(D); trG = float(np.trace(G))
    if trG <= 1e-15:
        return None
    P = list(players)
    cache = {frozenset(S): r2(G, trG, [b for p in S for b in players[p]])
             for k in range(len(P) + 1) for S in combinations(P, k)}
    phi = {c: sum(factorial(len(S)) * factorial(len(P) - len(S) - 1) / factorial(len(P))
                  * (cache[frozenset(S) | {c}] - cache[frozenset(S)])
                  for k in range(len(P)) for S in combinations([p for p in P if p != c], k)) for c in P}
    tot = cache[frozenset(P)]
    assert abs(sum(phi.values()) - tot) < 1e-9
    return phi, tot

def permanova_seq(D, blocks, n_perm=N_PERM, seed=0):
    n = len(D); G = gower(D); ss_tot = float(np.trace(G))
    if ss_tot <= 1e-15:
        return pd.DataFrame([dict(term="(no dispersion)", df=np.nan, R2=np.nan, F=np.nan, p=np.nan)])
    cols, Hs, ranks = [np.ones((n, 1))], [], [1]
    for _, B in blocks:
        cols.append(B); Q = orth(np.hstack(cols)); Hs.append(Q @ Q.T); ranks.append(Q.shape[1])
    df_res = n - ranks[-1]

    def stats(Gm):
        ss_res = ss_tot - float(np.vdot(Hs[-1], Gm)); prev, out = 0.0, []
        for i, H in enumerate(Hs):
            cur = float(np.vdot(H, Gm)); df = ranks[i + 1] - ranks[i]; ss = cur - prev; prev = cur
            out.append((ss, df, (ss / df) / (ss_res / df_res) if df > 0 and df_res > 0 and ss_res > 0 else np.nan))
        return out, ss_res

    obs, ss_res = stats(G)
    rng = np.random.default_rng(seed); exc = np.zeros(len(obs))
    for _ in range(n_perm):
        p = rng.permutation(n); res, _ = stats(G[np.ix_(p, p)])
        exc += np.array([(r[2] >= o[2] - 1e-12) if np.isfinite(o[2]) else 0 for r, o in zip(res, obs)])
    rows = [dict(term=nm, df=df, R2=ss / ss_tot, F=F, p=(1 + e) / (n_perm + 1) if df > 0 else np.nan)
            for (nm, _), (ss, df, F), e in zip(blocks, obs, exc)]
    rows.append(dict(term="Residual", df=df_res, R2=ss_res / ss_tot, F=np.nan, p=np.nan))
    return pd.DataFrame(rows)

# ---------------------------------------------------------------- load results ----
recs = []
for r_ in DESIGN.itertuples():
    zs = [np.load(job_path(r_.did, r_.pipe_id, s)) for s in range(CFG.n_train_seeds)]
    if any(str(z["status"]) != "ok" for z in zs):
        continue                                              # kNN + balanced holes
    Ns = [normalise(z["m"]) for z in zs]
    if any(x is None for x in Ns):
        continue                                              # constant-output pipelines
    y = DATA[r_.did].y[ROWS[r_.did]["train_idx"]]
    p_ = np.clip(zs[0]["oof"], 1e-15, 1 - 1e-15)
    loss = -(y * np.log(p_) + (1 - y) * np.log(1 - p_))
    recs.append(dict(did=r_.did, learner=r_.learner, balancing=r_.balancing, cfg=f"{r_.learner}#{r_.cfg_idx}",
                     logloss=loss.mean(), se=loss.std(ddof=1) / np.sqrt(len(loss)),
                     accuracy=float(((zs[0]["oof"] >= 0.5) == y).mean()), N=Ns))
POPF = pd.DataFrame(recs)
for did in CFG.dataset_ids:                                   # Rashomon sets relative to the best pipeline
    m = POPF.did == did
    b = POPF[m].logloss.idxmin()
    for c in (1, 2):
        POPF.loc[m, f"in_R_{c}se"] = POPF.loc[m, "logloss"] <= POPF.loc[b, "logloss"] + c * POPF.loc[b, "se"]
print("\npipelines analysed (kNN+balanced holes excluded):", POPF.groupby("did").size().to_dict())
print("Rashomon sets:", {did: {f"{c} SE": int(POPF[POPF.did == did][f"in_R_{c}se"].sum()) for c in (1, 2)}
                         for did in CFG.dataset_ids})
for did in CFG.dataset_ids:
    b = POPF.loc[POPF[POPF.did == did].logloss.idxmin()]
    print(f"best pipeline {did}: {b.learner} + {b.balancing}, log-loss {b.logloss:.4f}, accuracy {b.accuracy:.4f}")

# ---------------------------------------------------------------- analysis per dataset ----
RES, PERM = [], []
for did in CFG.dataset_ids:
    P = POPF[POPF.did == did].reset_index(drop=True)
    # delta_0: seed replicates under the shared explainer run
    d0 = {k: [] for k in DIST_NAMES}
    for N in P.N:
        Dd = pairwise_D(np.vstack([x[0] for x in N]), np.vstack([x[1] for x in N]))
        iu = np.triu_indices(len(N), 1)
        for k in DIST_NAMES:
            d0[k].append(Dd[k][iu].mean())
    d0 = {k: float(np.mean(v)) for k, v in d0.items()}
    # seed-0 distance matrix
    S0 = np.vstack([N[0][0] for N in P.N]); R0 = np.vstack([N[0][1] for N in P.N])
    D = pairwise_D(S0, R0)
    for set_name, mask in (("all pipelines", np.ones(len(P), bool)),
                           ("R_2se", P.in_R_2se.to_numpy(bool)), ("R_1se", P.in_R_1se.to_numpy(bool))):
        idx = np.where(mask)[0]
        if len(idx) < 3:
            continue
        sub = P.iloc[idx].reset_index(drop=True)
        for k in DIST_NAMES:
            Dk = D[k][np.ix_(idx, idx)]
            dbar = mean_pairwise(Dk)
            row = dict(did=did, set=set_name, distance=k, n=len(idx), n_learners=sub.learner.nunique(),
                       delta0=d0[k], D_bar=dbar, D_bar_over_delta0=safe_ratio(dbar, d0[k]))
            sh = shapley(Dk, {"learner": [dummies(sub.learner)], "balancing": [dummies(sub.balancing)]}) \
                if sub.learner.nunique() > 1 else None
            if sh:
                phi, tot = sh
                G = gower(Dk); trG = float(np.trace(G))
                inter = r2(G, trG, [dummies(sub.learner), dummies(sub.balancing),
                                    np.einsum("ni,nj->nij", dummies(sub.learner), dummies(sub.balancing))
                                    .reshape(len(sub), -1)]) - tot
                whole = shapley(Dk, {"learner_whole": [dummies(sub.cfg)], "balancing": [dummies(sub.balancing)]})
                row.update(R2_main=tot, learner_share=phi["learner"], balancing_share=phi["balancing"],
                           interaction=inter, learner_pct=safe_ratio(100 * phi["learner"], tot),
                           R2_whole=whole[1] if whole else np.nan,
                           learner_whole_pct=safe_ratio(100 * whole[0]["learner_whole"], whole[1]) if whole else np.nan)
            RES.append(row)
        if set_name == "all pipelines":                       # nested PERMANOVA (TV)
            t = permanova_seq(D["tv"], [("learner family", dummies(P.learner)),
                                        ("hyperparameters (within family)", dummies(P.cfg)),
                                        ("balancing", dummies(P.balancing))], seed=derive_seed("flaml-perm", did))
            PERM.append(t.assign(did=did))

    def mean_pairs(groups):
        dd = []
        for _, g in groups:
            ix = g.index.to_numpy()
            if len(ix) > 1:
                dd.extend(D["tv"][np.ix_(ix, ix)][np.triu_indices(len(ix), 1)])
        return safe_ratio(float(np.mean(dd)), d0["tv"]) if dd else np.nan

    RES.append(dict(did=did, set="paired effects (TV, delta_0 units)", distance="tv",
                    only_balancing=mean_pairs(P.groupby(["learner", "cfg"])),
                    only_hyperparameters=mean_pairs(P.groupby(["learner", "balancing"])),
                    only_learner=mean_pairs(P.groupby(["balancing"]))))

RES = pd.DataFrame(RES)
PERMT = pd.concat(PERM, ignore_index=True)
RES.to_csv(DIRS["metrics"] / "flaml_analysis.csv", index=False)
PERMT.to_csv(DIRS["metrics"] / "flaml_nested_permanova.csv", index=False)

pd.set_option("display.width", 250)
print("\nNoise floor delta_0 per distance:")
print(RES[(RES.set == "all pipelines")].pivot(index="did", columns="distance", values="delta0")[DIST_NAMES]
      .round(4).to_string())

tv = RES[(RES.distance == "tv") & (~RES.set.str.startswith("paired"))]
cols = [c for c in ["n", "n_learners", "delta0", "D_bar", "D_bar_over_delta0", "R2_main", "learner_pct",
                    "interaction", "learner_whole_pct"] if c in tv.columns]
print("\nFLAML — TV: noise floor, dispersion, Shapley shares (learner vs balancing):")
print(tv.set_index(["did", "set"])[cols].round(3).to_string())

alls = RES[(RES.set == "all pipelines") & RES.get("learner_share", pd.Series(dtype=float)).notna()]
print("\nLargest component per distance (all pipelines):",
      {(r.did, r.distance): ("learner" if r.learner_share > r.balancing_share else "balancing")
       for r in alls.itertuples()})

print("\nNested PERMANOVA (TV, all pipelines): family -> hyperparameters within family -> balancing")
print(PERMT.pivot_table(index="did", columns="term", values="R2").round(3).to_string())
print(PERMT.pivot_table(index="did", columns="term", values="p").round(4).to_string())

print("\nPaired effects (TV, delta_0 units; inf = noise floor is 0):")
print(RES[RES.set.str.startswith("paired")].set_index("did")[["only_learner", "only_hyperparameters", "only_balancing"]]
      .round(2).to_string())

# ---------------------------------------------------------------- side by side with auto-sklearn ----
AS = PROJECT / "results" / "clean" / "metrics"
try:
    as9 = pd.read_csv(AS / "item9_shapley_shares.csv")
    as3 = pd.read_csv(AS / "p2_item3_dispersion.csv")
    cmp_rows = []
    for did in CFG.dataset_ids:
        a = as9[(as9.did == did) & (as9.set == "all pipelines") & (as9.distance == "tv")].iloc[0]
        a3 = as3[(as3.did == did) & (as3.set == "all pipelines") & (as3.distance == "tv")].iloc[0]
        f = tv[(tv.did == did) & (tv.set == "all pipelines")].iloc[0]
        cmp_rows.append(dict(did=did, system="auto-sklearn", D_bar=a3["D_bar"], D_bar_over_delta0=a3["D_bar/delta0"],
                             classifier_pct=100 * a.phi_classifier / a.R2_main,
                             preprocessor_pct=100 * a.phi_preprocessor / a.R2_main,
                             balancing_pct=100 * a.phi_balancing / a.R2_main))
        cmp_rows.append(dict(did=did, system="FLAML", D_bar=f.D_bar, D_bar_over_delta0=f.D_bar_over_delta0,
                             classifier_pct=f.get("learner_pct", np.nan), preprocessor_pct=np.nan,
                             balancing_pct=100 - f.get("learner_pct", np.nan)))
    CMPF = pd.DataFrame(cmp_rows).set_index(["did", "system"])
    print("\nauto-sklearn vs FLAML (TV, all pipelines; shares of explained variance):")
    print(CMPF.round(3).to_string())
    CMPF.to_csv(DIRS["metrics"] / "flaml_vs_autosklearn.csv")
except Exception as e:
    print("\n(auto-sklearn comparison skipped:", type(e).__name__, str(e)[:120], ")")

# ---------------------------------------------------------------- figure ----
fig, axes = plt.subplots(1, len(CFG.dataset_ids), figsize=(5 * len(CFG.dataset_ids), 4), sharey=True)
for ax, did in zip(np.atleast_1d(axes), CFG.dataset_ids):
    t = tv[tv.did == did].set_index("set")
    sets = [s for s in ("all pipelines", "R_2se", "R_1se") if s in t.index and pd.notna(t.loc[s].get("R2_main"))]
    for i, s in enumerate(sets):
        ax.bar(i, t.loc[s, "learner_share"], color="#3b6ea8", label="learner" if i == 0 else None)
        ax.bar(i, t.loc[s, "balancing_share"], bottom=t.loc[s, "learner_share"], color="#6aa84f",
               label="balancing" if i == 0 else None)
        ax.bar(i, t.loc[s, "interaction"], bottom=t.loc[s, "R2_main"], color="lightgrey", hatch="//",
               label="learner x balancing" if i == 0 else None)
    ax.set_xticks(range(len(sets))); ax.set_xticklabels(sets); ax.set_title(f"FLAML, dataset {did}")
np.atleast_1d(axes)[0].set_ylabel("R² (Shapley shares, TV)")
np.atleast_1d(axes)[0].legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(DIRS["figures"] / "flaml_shapley_shares.png", dpi=200)
print("\nsaved:", DIRS["metrics"] / "flaml_analysis.csv", "|", DIRS["figures"] / "flaml_shapley_shares.png")

Seed check: share of pipelines whose 3 training seeds give the IDENTICAL model
(1.00 = seed has no effect, 0.00 = seeds change the model)
did              31   37
learner                 
catboost        0.0  0.0
extra_tree      0.0  0.0
kneighbor       1.0  1.0
lgbm            0.2  0.0
lrl1            0.0  0.0
lrl2            1.0  1.0
rf              0.0  0.0
xgb_limitdepth  0.7  0.4
xgboost         0.2  0.0

pipelines analysed (kNN+balanced holes excluded): {31: 73, 37: 79}
Rashomon sets: {37: {'1 SE': 17, '2 SE': 29}, 31: {'1 SE': 10, '2 SE': 22}}
best pipeline 37: catboost + none, log-loss 0.4817, accuracy 0.7523
best pipeline 31: extra_tree + none, log-loss 0.5103, accuracy 0.7443

Noise floor delta_0 per distance:
distance      tv    jac3    jac5     rbo
did                                     
31        0.0685  0.1635  0.1771  0.0736
37        0.0563  0.0814  0.0719  0.0353

FLAML — TV: noise floor, dispersion, Shapley shares (learner vs balancing):
                      n  n_le